# Ingest races.csv file
1. Read the file using spark dataframe reader API
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/02.Ayuda_Bronze

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
archivo_fuente = f"{ruta}/races.csv"
nombre_tabla = f"{catalogo}.{esquema_bronze}.races"

#### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

esquema_races = StructType([
    StructField('season',   IntegerType()),
    StructField("round",    IntegerType()),
    StructField("url",      StringType()),
    StructField("raceName", StringType()),
    StructField("date",     DateType()),
    StructField("circuitId", StringType())
])

In [0]:
df_races = (
    spark.read
         .format('csv')
         .option('header', 'true')
         .option('mode', 'FAILFAST')
         .schema(esquema_races)
         .load(archivo_fuente)
)

In [0]:
display(df_races)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
df_final_races = add_ingesta_metadata(df_races)

In [0]:
display(df_final_races)

#### Step 3 - Write to bronze delta table

In [0]:
(
    df_final_races
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(nombre_tabla)
)

In [0]:
display(spark.table(nombre_tabla))